## **Fine-Tuning a Small Language Model (SLM) Using Transformer Models**


These libraries are required for loading datasets, fine-tuning transformer models, and evaluating performance.



In [1]:
!pip install transformers datasets accelerate evaluate torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


Import libraries

In [20]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
import evaluate

The IMDB dataset contains movie reviews labeled as positive or negative. It is suitable for text-based language model fine-tuning.


In [6]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

Small Language Model (distilgpt2)

In [4]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Tokenize the dataset


In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id



Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
        return_attention_mask=True
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Prepare Training and Evaluation Data

In [13]:
train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(5000))
eval_dataset = tokenized_dataset["test"].shuffle(seed=42).select(range(1000))


Define Training Arguments

In [16]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    report_to="tensorboard",
    save_total_limit=1
)


Create Trainer

In [21]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

Fine-Tune the Model

In [24]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,3.764893,3.830621
2,3.701986,3.837063


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2500, training_loss=3.7349734375, metrics={'train_runtime': 487.7389, 'train_samples_per_second': 20.503, 'train_steps_per_second': 5.126, 'total_flos': 326620938240000.0, 'train_loss': 3.7349734375, 'epoch': 2.0})

Evaluate the Model

In [25]:
eval_results = trainer.evaluate()
eval_results


{'eval_loss': 3.8370625972747803,
 'eval_runtime': 9.6171,
 'eval_samples_per_second': 103.981,
 'eval_steps_per_second': 25.995,
 'epoch': 2.0}

Lower perplexity indicates better language modeling performance.

In [26]:
import math
perplexity = math.exp(eval_results["eval_loss"])
print("Perplexity:", perplexity)


Perplexity: 46.389010906675736


Observations:
- The training loss decreased across epochs, indicating effective learning.
- The model showed improved text generation coherence after fine-tuning.
- Perplexity reduced compared to the base model, confirming performance improvement.

Limitations:
- Limited dataset size due to Colab constraints.
- More epochs and larger datasets could further improve results.
